# Pilot: Perception & Reasoning Entropy Signal Check (FERMAT + Qwen2.5-VL)

Purpose: the smallest, fastest end-to-end run to check whether perception entropy
(instability in transcribing handwritten math) and reasoning entropy (instability in
grading an answer for errors) are visibly higher on items the model gets wrong. This
is a scoped-down pilot, not the full study — no bootstrap CIs, no baseline suite, no
conformal calibration, no human double grading.

**This notebook does not carry its own copy of the pipeline logic.** Every non-GPU
piece (data loading/filtering, prompt formatting, output parsing, entropy calculation)
lives in the `pilot/` package and is unit-tested locally. This notebook clones that
same code repo fresh each session and installs it, so there is a single source of
truth — editing a module locally and re-running this notebook next session picks up
the change automatically, with no manual copy/paste re-sync step.

Only the GPU-dependent parts live here: installing GPU deps, loading the model, and
running the sampling loop.

In [17]:
# Install cell: GPU-dependent packages only.
# `datasets` is intentionally also in the local requirements.txt -- each
# environment installs its own copy independently, no conflict.
# torch is not installed explicitly: Colab GPU runtimes ship with it preinstalled.
!pip install -q transformers accelerate bitsandbytes datasets qwen-vl-utils

In [ ]:
# Auth & code/results access cell.
import json
import os
from getpass import getpass

from huggingface_hub import login

# --- Drive mount first: it holds both the model cache and the token store ---
from google.colab import drive

drive.mount("/content/drive")
PROJECT_DIR = "/content/drive/MyDrive/uncertainty-math-vlm"
DRIVE_MODEL_CACHE = f"{PROJECT_DIR}/model_cache"
os.makedirs(DRIVE_MODEL_CACHE, exist_ok=True)

# --- Tokens: entered ONCE, then cached on your Drive ---
# Deliberately not hardcoded in this notebook. This file is tracked in a
# public repo, and GitHub's secret scanning auto-revokes any ghp_ token that
# lands in a public commit -- so an inline token would stop working by
# itself. Drive is private to your account, survives runtime recycling, and
# git never touches it, so you get the same "no retyping" result safely.
TOKEN_FILE = f"{PROJECT_DIR}/.tokens.json"
RESET_TOKENS = False  # set True once to replace previously saved tokens


def get_token(name, prompt):
    """Return a saved token, prompting (once) and persisting it if absent."""
    tokens = {}
    if os.path.exists(TOKEN_FILE):
        with open(TOKEN_FILE) as f:
            tokens = json.load(f)
    if RESET_TOKENS or not tokens.get(name):
        tokens[name] = getpass(prompt).strip()
        with open(TOKEN_FILE, "w") as f:
            json.dump(tokens, f)
        os.chmod(TOKEN_FILE, 0o600)
        print(f"Saved {name} to Drive -- you will not be asked for it again.")
    return tokens[name]


HF_TOKEN = get_token("HF_TOKEN", "Hugging Face token (asked once): ")
GH_TOKEN = get_token("GH_TOKEN", "GitHub token with 'repo' scope (asked once): ")

if not HF_TOKEN.startswith("hf_"):
    raise ValueError(
        "Stored Hugging Face token does not start with 'hf_'. Set "
        "RESET_TOKENS = True and re-run this cell to replace it."
    )

login(token=HF_TOKEN)
print("Hugging Face login OK")

# --- Clone the repo (code + results live in the same repo for this pilot) ---
# Cloned anonymously: the repo is public, so read access needs no token, and
# keeping the token out of the clone URL means a clone error can never echo
# it into this notebook's saved output. The token is used only to push.
REPO_URL = "https://github.com/sepehrmaleki369/uncertainty-math-vlm.git"

# Remove any stale clone from a previous (possibly failed) run so this cell
# is safe to re-run -- git clone silently no-ops into a pre-existing
# directory, which would otherwise leave `repo/` incomplete without error.
!rm -rf repo
!git clone -q {REPO_URL} repo

# %pip (not !pip) installs into the *running kernel's* environment -- !pip
# can silently target a different Python install.
%pip install -q -e repo/

# An editable install writes an `__editable__.pilot-*.pth` file into
# site-packages, but .pth files are only processed by the `site` module at
# INTERPRETER STARTUP. The kernel is already running, so it never sees them
# and `import pilot` fails with ModuleNotFoundError even though the install
# reported success. Putting the repo on sys.path directly makes the package
# importable right now, with no kernel restart needed.
import importlib
import sys

REPO_DIR = os.path.abspath("repo")
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
importlib.invalidate_caches()

import pilot.data
import pilot.prompts
import pilot.parsing
import pilot.entropy

print(f"pilot package imported from: {os.path.dirname(pilot.__file__)}")

In [19]:
# Model load cell.
# Start with the 3B model for the first smoke test -- same prompt format and
# code path as the 7B, but noticeably faster to load and run, so early bugs
# get caught cheaply. Swap MODEL_ID to the 7B line below once the pipeline
# runs cleanly end to end on the 3B.
import torch
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
# MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"  # swap in once the 3B pipeline is clean

# If memory is tight on the 7B, load in 4-bit instead:
# from transformers import BitsAndBytesConfig
# quantization_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.bfloat16,
# )
# model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
#     MODEL_ID,
#     quantization_config=quantization_config,
#     device_map="auto",
#     cache_dir=DRIVE_MODEL_CACHE,
# )

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    cache_dir=DRIVE_MODEL_CACHE,
)
processor = AutoProcessor.from_pretrained(MODEL_ID, cache_dir=DRIVE_MODEL_CACHE)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

In [ ]:
# Sampling loop cell.
#
# K=5 samples at temperature=0.7 for each of the transcription and grading
# prompts, plus 2 samples at temperature=0 (greedy, back-to-back) as a
# low-temperature sanity anchor -- 2 draws, not 1, because a single-sample
# entropy is mathematically zero by definition and could never catch a real
# bug (e.g. an accidental sampling flag, batching nondeterminism). With 2
# draws it's a genuine, if noisy, instability check.
#
# Each generation call is wrapped in a bounded retry scoped strictly to
# infrastructure-level failures (dropped connection, transient OOM, Colab
# runtime hiccup) -- never around parsing. A response that fails to parse is
# real data about that sample's behavior under that prompt, not a transient
# failure to retry past; retrying until a clean parse appears would silently
# bias every entropy estimate downward.
#
# Results are checkpointed to Drive after EVERY item. /content is ephemeral,
# so anything kept only in memory or in /content is lost when the runtime
# disconnects or recycles -- which on a 30-90 minute run is a real risk. Re-run
# this cell after a disconnect and it resumes from the last completed item.
import gc
import json
import time

import torch
from qwen_vl_utils import process_vision_info
from tqdm.auto import tqdm

# This loop makes 14 model calls per item (2 prompts x (5 sampled + 2 greedy)),
# so N=25 is ~350 calls and can take a while on a T4. Consider setting N=2 for
# a first end-to-end check that the whole pipeline runs, then raising it.
N = 25  # smoke-test size; bump to 75-100 for the Step 3 7B run
SEED = 42
K = 5
TEMP = 0.7
N_TEMP0 = 2
MAX_RETRIES = 3
RETRY_PAUSE_SECONDS = 5

# Only these fields are needed downstream (cell 6 never touches the image),
# and dropping the PIL image keeps each checkpoint row JSON-serializable.
META_FIELDS = ("orig_q", "pert_a", "has_error", "handwriting_style", "image_quality")

INFRA_EXCEPTIONS = (
    ConnectionError,
    TimeoutError,
    torch.cuda.OutOfMemoryError,
    OSError,
)


def generate(messages, do_sample: bool, temperature: float | None):
    """Run one generation call, retrying only on infrastructure-level failures."""
    last_exc = None
    for attempt in range(MAX_RETRIES):
        try:
            text_prompt = processor.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            image_inputs, video_inputs = process_vision_info(messages)
            inputs = processor(
                text=[text_prompt],
                images=image_inputs,
                videos=video_inputs,
                padding=True,
                return_tensors="pt",
            ).to(model.device)

            gen_kwargs = {"max_new_tokens": 512, "do_sample": do_sample}
            if do_sample:
                gen_kwargs["temperature"] = temperature

            with torch.no_grad():
                output_ids = model.generate(**inputs, **gen_kwargs)

            trimmed = output_ids[:, inputs["input_ids"].shape[1]:]
            return processor.batch_decode(
                trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=True
            )[0]
        except INFRA_EXCEPTIONS as exc:
            last_exc = exc
            gc.collect()
            torch.cuda.empty_cache()
            if attempt < MAX_RETRIES - 1:
                time.sleep(RETRY_PAUSE_SECONDS)
    raise last_exc


def run_batch(messages, n, do_sample, temperature, stage, item_idx, n_items, pbar):
    """Draw n samples for one prompt, ticking the progress bar after each call."""
    outputs = []
    for j in range(n):
        pbar.set_postfix_str(f"item {item_idx + 1}/{n_items} | {stage} {j + 1}/{n}")
        outputs.append(generate(messages, do_sample=do_sample, temperature=temperature))
        pbar.update(1)
    return outputs


sample = pilot.data.load_fermat_sample(n=N, seed=SEED)
n_items = len(sample)

# Checkpoint file is keyed by model/N/seed so a different config starts fresh
# rather than silently resuming from an unrelated run.
CHECKPOINT_DIR = "/content/drive/MyDrive/uncertainty-math-vlm/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
checkpoint_path = (
    f"{CHECKPOINT_DIR}/raw_{MODEL_ID.split('/')[-1]}_n{N}_seed{SEED}.jsonl"
)

raw_results = []
if os.path.exists(checkpoint_path):
    with open(checkpoint_path) as f:
        raw_results = [json.loads(line) for line in f if line.strip()]
    print(f"Found checkpoint: resuming after {len(raw_results)} completed items")
    print(f"  {checkpoint_path}")

n_done = len(raw_results)
if n_done >= n_items:
    print(f"All {n_items} items already done -- nothing to generate.")
else:
    calls_per_item = 2 * (K + N_TEMP0)
    remaining_calls = (n_items - n_done) * calls_per_item
    print(
        f"{n_items - n_done} items left x {calls_per_item} calls "
        f"= {remaining_calls} generation calls"
    )

    with tqdm(total=remaining_calls, desc="generating", unit="call") as pbar:
        for item_idx, item in enumerate(sample):
            if item_idx < n_done:
                continue  # already checkpointed by an earlier run

            image = item["image"]
            transcription_messages = pilot.prompts.build_transcription_messages(image)
            grading_messages = pilot.prompts.build_grading_messages(image)

            entry = {
                "item": {k: item[k] for k in META_FIELDS},
                "transcription_samples_raw": run_batch(
                    transcription_messages, K, True, TEMP,
                    "transcribe T=0.7", item_idx, n_items, pbar,
                ),
                "grading_samples_raw": run_batch(
                    grading_messages, K, True, TEMP,
                    "grade T=0.7", item_idx, n_items, pbar,
                ),
                "transcription_temp0_raw": run_batch(
                    transcription_messages, N_TEMP0, False, None,
                    "transcribe T=0", item_idx, n_items, pbar,
                ),
                "grading_temp0_raw": run_batch(
                    grading_messages, N_TEMP0, False, None,
                    "grade T=0", item_idx, n_items, pbar,
                ),
            }
            raw_results.append(entry)

            # Append-only, flushed immediately: a disconnect mid-run costs at
            # most the item in flight, not the whole run.
            with open(checkpoint_path, "a") as f:
                f.write(json.dumps(entry, default=str) + "\n")
                f.flush()

print(f"Collected raw samples for {len(raw_results)} items.")
print(f"Checkpoint on Drive: {checkpoint_path}")

### Scoring cell — note on the temperature-0 anchor

`temp0_entropy_transcription` / `temp0_entropy_grading` below are computed over
**2** greedy draws per item, not 1. With only 1 sample, `cluster_entropy` would be
mathematically zero by definition regardless of anything the model actually did —
a tautology, not a finding. With 2 draws, a nonzero value is a real (if noisy)
signal of low-temperature instability — e.g. an accidentally-enabled sampling flag,
or batching nondeterminism — and should be read as a **pipeline smoke test**, not
as an empirical claim about the model's true low-temperature behavior. If these are
not at or near zero for nearly every item, something in the sampling or parsing is
broken and needs fixing before trusting anything else.

Per-item parse-failure counts (`n_transcription_parse_failures`,
`n_grading_parse_failures`, out of K=5) are also recorded here so Step 4 can check
the **aggregate** parse-failure rate across the whole run — a systematic regex or
prompt-format bug can make every sample in an item fail to parse, which collapses
to a single confident `<PARSE_FAILURE>` cluster (entropy 0) and would otherwise
hide inside numbers that look clean.

In [21]:
# Scoring cell.
import pilot.parsing
import pilot.entropy

scored_results = []
for entry in raw_results:
    item = entry["item"]

    transcription_parsed = [
        pilot.parsing.parse_transcription(t) for t in entry["transcription_samples_raw"]
    ]
    grading_parsed = [pilot.parsing.parse_grading(t) for t in entry["grading_samples_raw"]]
    transcription_temp0_parsed = [
        pilot.parsing.parse_transcription(t) for t in entry["transcription_temp0_raw"]
    ]
    grading_temp0_parsed = [pilot.parsing.parse_grading(t) for t in entry["grading_temp0_raw"]]

    perception_entropy = pilot.entropy.cluster_entropy(transcription_parsed)
    reasoning_entropy = pilot.entropy.cluster_entropy(
        [None if d is None else str(d) for d in grading_parsed]
    )
    temp0_entropy_transcription = pilot.entropy.cluster_entropy(transcription_temp0_parsed)
    temp0_entropy_grading = pilot.entropy.cluster_entropy(
        [None if d is None else str(d) for d in grading_temp0_parsed]
    )

    majority_transcription, _ = pilot.entropy.majority_cluster(transcription_parsed)
    transcription_correct = majority_transcription == pilot.entropy.normalize_string(
        item["pert_a"]
    )

    majority_grading, _ = pilot.entropy.majority_cluster(
        [None if d is None else str(d) for d in grading_parsed]
    )
    grading_correct = majority_grading == pilot.entropy.normalize_string(str(item["has_error"]))

    n_transcription_parse_failures = sum(1 for t in transcription_parsed if t is None)
    n_grading_parse_failures = sum(1 for d in grading_parsed if d is None)

    scored_results.append(
        {
            "orig_q": item["orig_q"],
            "pert_a": item["pert_a"],
            "has_error": item["has_error"],
            "handwriting_style": item["handwriting_style"],
            "image_quality": item["image_quality"],
            "perception_entropy": perception_entropy,
            "reasoning_entropy": reasoning_entropy,
            "temp0_entropy_transcription": temp0_entropy_transcription,
            "temp0_entropy_grading": temp0_entropy_grading,
            "transcription_correct": transcription_correct,
            "grading_correct": grading_correct,
            "n_transcription_parse_failures": n_transcription_parse_failures,
            "n_grading_parse_failures": n_grading_parse_failures,
            "all_transcription_samples_raw": entry["transcription_samples_raw"],
            "all_grading_samples_raw": entry["grading_samples_raw"],
            "temp0_transcription_raw": entry["transcription_temp0_raw"],
            "temp0_grading_raw": entry["grading_temp0_raw"],
            "model_id": MODEL_ID,
            "n_items": N,
        }
    )

print(f"Scored {len(scored_results)} items.")

Scored 25 items.


In [1]:
# Save cell: write CSV into the cloned repo's results/ dir, commit, push.
import subprocess
from datetime import datetime, timezone
from getpass import getpass

import pandas as pd

df = pd.DataFrame(scored_results)

model_slug = MODEL_ID.split("/")[-1].lower().replace(".", "")
timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
csv_name = f"results_{model_slug}_{timestamp}.csv"

os.makedirs("repo/results", exist_ok=True)
csv_path = f"repo/results/{csv_name}"
df.to_csv(csv_path, index=False)
print(f"Wrote {csv_path} ({len(df)} rows)")

# Write a copy to Drive BEFORE touching git. Pushing can fail for auth or
# fast-forward reasons, and losing an hour of GPU output to a git problem
# would be painful -- this copy survives regardless of what happens below.
drive_results = "/content/drive/MyDrive/uncertainty-math-vlm/results"
os.makedirs(drive_results, exist_ok=True)
df.to_csv(f"{drive_results}/{csv_name}", index=False)
print(f"Backup copy written to {drive_results}/{csv_name}")

# Secrets to scrub from any git output we print. A failed push makes git
# echo the remote URL back in its error message, which would otherwise
# print the token straight into this notebook's saved output.
_REDACT = []


def git(*args):
    """Run a git command in repo/, surfacing output (redacted) when it fails."""
    result = subprocess.run(
        ["git", "-C", "repo", *args], capture_output=True, text=True
    )
    output = (result.stdout or "") + (result.stderr or "")
    for secret in _REDACT:
        if secret:
            output = output.replace(secret, "***")
    if result.returncode != 0 and output.strip():
        print(output.strip())
    return result


# A fresh clone has no committer identity, so `git commit` fails with exit
# 128 ("Please tell me who you are"). Set it for this clone only.
git("config", "user.email", "colab-pilot@localhost")
git("config", "user.name", "Colab Pilot Run")

git("add", f"results/{csv_name}")
commit = git("commit", "-m", f"Add pilot results: {csv_name}")
if commit.returncode != 0:
    raise RuntimeError("git commit failed -- see output above")
print(f"Committed {csv_name}")

# Pushing requires credentials even for a PUBLIC repo: anonymous HTTPS is
# read-only. Reuse the token from the auth cell if the repo was private,
# otherwise prompt for one now (needs 'repo' / contents:write scope).
GH_PUSH_TOKEN = (globals().get("GH_TOKEN") or "").strip()
if not GH_PUSH_TOKEN:
    GH_PUSH_TOKEN = getpass("GitHub token (to push results), then press Enter: ").strip()
_REDACT.append(GH_PUSH_TOKEN)

if not GH_PUSH_TOKEN:
    print("No token given -- skipping push. CSV is saved on Drive and in repo/results/.")
else:
    push_url = REPO_URL.replace("https://", f"https://{GH_PUSH_TOKEN}@")

    # The clone may be behind if anything was pushed from elsewhere since
    # this session started; rebase our new commit on top before pushing.
    if git("fetch", push_url, "main").returncode == 0:
        if git("rebase", "FETCH_HEAD").returncode != 0:
            git("rebase", "--abort")
            print("Rebase onto remote failed; attempting push anyway.")

    if git("push", push_url, "HEAD:main").returncode == 0:
        print("Pushed results to the repo.")
    else:
        print(
            "Push failed (see output above). The CSV is safe on Drive and in "
            "repo/results/ -- you can retry the push without re-running the model."
        )

NameError: name 'scored_results' is not defined